
noise
を
生成したいデータ分布(data)
とする。


Flow Matching では通常、

$$
t=0:\ \text{noise}
$$

$$
t=1:\ \text{data}
$$

と置く。

線形 path なら、

$$
x_t=(1-t)x_0+t x_1
$$

ここで

$$
x_0\sim \mathcal N(0,I)
$$

$$
x_1\sim p_{\mathrm{data}}(x)
$$

である。

したがって端点は、

$$
x_{t=0}=x_0
$$

$$
x_{t=1}=x_1
$$

すなわち、

$$
x_{t=0}\sim \mathcal N(0,I)
$$

$$
x_{t=1}\sim p_{\mathrm{data}}(x)
$$

である。


Flow Matching の最も基本的なアルゴリズム（Conditional Flow Matching, Linear Path）は以下です。


## 1. データをサンプリング

データ分布から

$$
x_1 \sim p_{\mathrm{data}}(x)
$$

をサンプリングする。


## 2. ノイズをサンプリング

単純な事前分布から

$$
x_0 \sim \mathcal N(0,I)
$$

をサンプリングする。


## 3. 時刻をサンプリング

$$
t \sim U(0,1)
$$

をサンプリングする。


## 4. Probability Path を作る

線形補間を用いて

$$
x_t=(1-t)x_0+t x_1
$$

を計算する。



## 5. 正解速度を計算

線形 path の速度は

$$
u_t(x_t)
=
\frac{d x_t}{dt}
=
x_1-x_0
$$

である。

したがって教師信号は

$$
u_{\mathrm{target}}
=
x_1-x_0
$$

となる。


## 6. Neural Network に学習させる

速度場

$$
u_\theta(x_t,t)
$$

を学習する。

損失関数は

$$
L
=
E
\left[
\left|
u_\theta(x_t,t)
-
(x_1-x_0)
\right|^2
\right]
$$

である。

## Training Algorithm

```text
repeat

    sample x1 ~ pdata

    sample x0 ~ N(0,I)

    sample t ~ Uniform(0,1)

    xt = (1-t)x0 + t x1

    vtarget = x1 - x0

    loss =
        ||uθ(xt,t)-vtarget||²

    update θ

until convergence
```

---

## Sampling Algorithm

学習後は

$$
x_{t=0}
\sim
\mathcal N(0,I)
$$

から開始し、

ODE

$$
\frac{dx}{dt}
=
u_\theta(x,t)
$$

を解く。

Euler 法なら

$$
x_{t+\Delta t}
=
x_t
+
u_\theta(x_t,t)\Delta t
$$

で更新する。



## Sampling Pseudocode

```text
x = sample N(0,I)

for t=0 → 1:
    x = x + uθ(x,t) Δt

return x
```



Flow Matching を一言で書くと

```text
Training:
    ノイズ点 x0 と
    データ点 x1 を結ぶ速度を学習

Sampling:
    学習した速度場を積分して
    ノイズ分布をデータ分布へ輸送
```

です。

Diffusion Model が

```text
noise prediction
↓
reverse diffusion
```

を学習するのに対し、

Flow Matching は

```text
velocity prediction
↓
ODE integration
```

を学習します。

以下は、arXivの2412.06264v1の1D bimodal distribution 例をかなり簡略化した **最小 Flow Matching Python script** です。直線 path

$$
x_t=(1-t)x_0+t x_1,\qquad v=x_1-x_0
$$

を学習します。

Ref. Yaron Lipman, Marton Havasi, Peter Holderrieth, Neta Shaul, Matt Le, Brian Karrer, Ricky T. Q. Chen, David Lopez-Paz, Heli Ben-Hamu, Itai Gat, "Flow Matching Guide and Code", arXiv: 2412.06264v1


以下を行う生成モデルを作成する。
```
Gaussian distribution
        ↓
   Generative Model
        ↓
簡単な分布
```


In [ ]:
# GELUとは？
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-5, 5, 1000)

def gelu_torch(x):
    return (
        0.5 * x *
        (
            1.0
            + np.tanh(
                np.sqrt(2/np.pi)
                * (x + 0.044715*x**3)
            )
        )
    )

plt.plot(x, gelu_torch(x))
plt.title("PyTorch GELU")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# -----------------------
# setup
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)

# -----------------------
# target distribution: 1D bimodal Gaussian mixture
# -----------------------
def sample_target(n):
    # mixture: 55% N(-0.85, 0.65^2), 45% N(1.5, 0.25^2)
    comp = np.random.rand(n) > 0.55
    mu = np.where(comp, 1.5, -0.85)
    sigma = np.where(comp, 0.25, 0.65)
    x = np.random.normal(mu, sigma)
    return torch.tensor(x, dtype=torch.float32).view(-1, 1).to(device)

# -----------------------
# velocity model u_theta(x_t, t)
# -----------------------
class FlowModel(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),   # input: [x_t, t]
            nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),   # output: velocity
        )

    def forward(self, x_t, t):
        return self.net(torch.cat([x_t, t], dim=1))

model = FlowModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# -----------------------
# training
# -----------------------
batch_size = 256
n_iter = 5000
losses = []

for step in range(n_iter):
    x0 = torch.randn(batch_size, 1, device=device)  # Gaussian noise
    x1 = sample_target(batch_size)                  # data
    t = torch.rand(batch_size, 1, device=device)

    x_t = (1 - t) * x0 + t * x1
    v_target = x1 - x0

    v_pred = model(x_t, t)
    loss = ((v_pred - v_target) ** 2).mean()

    opt.zero_grad()
    loss.backward()
    opt.step()

    losses.append(loss.item())

    if step % 500 == 0:
        print(step, loss.item())

![fig/1510_training.png](fig/1510_training.png)

In [ ]:
# -----------------------
# sampling by Euler integration
# -----------------------
@torch.no_grad()
def sample(model, n=10000, n_steps=100):
    x = torch.randn(n, 1, device=device)
    ts = torch.linspace(0, 1, n_steps + 1, device=device)

    for i in range(n_steps):
        t = ts[i].expand(n, 1)
        dt = ts[i + 1] - ts[i]
        x = x + model(x, t) * dt

    return x.cpu().numpy().ravel()

x_gen = sample(model)

画像にするには、$v = \frac{dx}{dt}$を$x_{t=1}$に変換する必要がある。

![fig/1510_generation.png](fig/1510_generation.png)

In [ ]:
# -----------------------
# plot
# -----------------------
plt.figure(figsize=(8, 3))
plt.hist(x_gen, bins=100, density=True, alpha=0.6, label="generated")

x_true = sample_target(10000).cpu().numpy().ravel()
plt.hist(x_true, bins=100, density=True, alpha=0.4, label="target")

plt.legend()
plt.xlabel("x")
plt.ylabel("density")
plt.title("1D Flow Matching: target vs generated")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("training loss")
plt.tight_layout()
plt.show()

学習時は ODE(常微分方程式） を解かずに v_target = x1 - x0 を教師信号として回帰し、生成時だけ x = x + u_theta(x,t) dt で積分することです。

In [ ]:
@torch.no_grad()
def sample_paths(model, n_samples=100000, n_steps=200):
    """
    全サンプルの軌跡を保存
    returns:
        paths.shape = (n_samples, n_steps+1)
    """
    x = torch.randn(n_samples, 1, device=device)

    paths = torch.zeros(
        n_samples,
        n_steps + 1,
        device=device
    )

    paths[:, 0] = x[:, 0]

    ts = torch.linspace(0, 1, n_steps + 1, device=device)

    for i in range(n_steps):
        t = ts[i].expand(n_samples, 1)
        dt = ts[i + 1] - ts[i]

        x = x + model(x, t) * dt

        paths[:, i + 1] = x[:, 0]

    return paths.cpu().numpy()

In [ ]:
# -----------------------
# Path density + velocity arrows
# -----------------------

paths = sample_paths(
    model,
    n_samples=50000,
    n_steps=200
)

n_steps = paths.shape[1] - 1

x_min = -3
x_max = 3
n_bins = 300

density = np.zeros((n_steps + 1, n_bins))
edges = np.linspace(x_min, x_max, n_bins + 1)

for i in range(n_steps + 1):
    density[i] = np.histogram(paths[:, i], bins=edges)[0]

# plot density
plt.figure(figsize=(9, 5))

plt.imshow(
    density.T,
    origin="lower",
    aspect="auto",
    extent=[0, 1, x_min, x_max],
    cmap="viridis"
)

plt.colorbar(label="count")
plt.xlabel("t")
plt.ylabel("x")
plt.title("Path Density with Flow Velocity Field")

# -----------------------
# velocity arrows
# -----------------------

# 矢印を描く格子
t_grid = np.linspace(0, 1, 15)
x_grid = np.linspace(x_min, x_max, 25)

T, X = np.meshgrid(t_grid, x_grid)

# model に入れる形へ
t_tensor = torch.tensor(
    T.reshape(-1, 1),
    dtype=torch.float32,
    device=device
)

x_tensor = torch.tensor(
    X.reshape(-1, 1),
    dtype=torch.float32,
    device=device
)

with torch.no_grad():
    V = model(x_tensor, t_tensor).cpu().numpy().reshape(X.shape)

# 横方向は dt 方向なので一定
U = np.ones_like(V)

# 矢印が長すぎると見づらいので正規化
norm = np.sqrt(U**2 + V**2)
U_plot = U / norm
V_plot = V / norm

plt.quiver(
    T,
    X,
    U_plot,
    V_plot,
    color="white",
    angles="xy",
    scale_units="xy",
    scale=25,
    width=0.003,
    alpha=0.8
)

plt.tight_layout()
plt.show()


$$
(t,x)↦(t+Δt,x+u_θ(x,t)Δt)
$$
という変化をします。

縦縦軸が x で、$\Delta t=1$として
```
U = 1
V = u_theta(x, t)
```
として矢印を書いています。